In [14]:
import pandas as pd
import requests

In [15]:
BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1/datasets"

def get_actuals(start: str, end: str) -> pd.DataFrame:
    """Generación eólica real (FUELHH, fuelType=WIND). Fechas 'YYYY-MM-DD'."""
    r = requests.get(
        f"{BASE_URL}/FUELHH/stream",
        params={
            "settlementDateFrom": start,
            "settlementDateTo": end,
            "fuelType": "WIND",
        },
        timeout=120,
    )
    r.raise_for_status()
    df = pd.DataFrame(r.json())
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True)
    return df[["startTime", "generation"]]


def get_forecasts(start: str, end: str) -> pd.DataFrame:
    """Pronósticos eólicos (WINDFOR). Fechas ISO, ej. '2025-12-20T00:00Z'."""
    r = requests.get(
        f"{BASE_URL}/WINDFOR/stream",
        params={
            "publishDateTimeFrom": start,
            "publishDateTimeTo": end,
        },
        timeout=120,
    )
    r.raise_for_status()
    df = pd.DataFrame(r.json())
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True)
    df["publishTime"] = pd.to_datetime(df["publishTime"], utc=True)
    return df[["startTime", "publishTime", "generation"]]

In [16]:
actual = get_actuals("2026-01-01", "2026-02-05")
fc = get_forecasts("2025-12-20T00:00Z", "2026-02-01T00:00Z")

print(actual["startTime"].min(), actual["startTime"].max(), len(actual))
print(fc["publishTime"].min(), fc["publishTime"].max(), len(fc))
print(fc["startTime"].min(), fc["startTime"].max())

2026-01-01 00:00:00+00:00 2026-02-05 23:30:00+00:00 1728
2025-12-20 02:30:00+00:00 2026-01-31 22:30:00+00:00 25112
2025-12-19 21:00:00+00:00 2026-02-02 21:00:00+00:00


In [17]:
fc["horizon_h"] = (fc["startTime"] - fc["publishTime"]).dt.total_seconds() / 3600

print(fc["horizon_h"].describe())                      # horizonte mín/máx
print("horizonte negativo:", (fc["horizon_h"] < 0).sum())
print(fc["startTime"].drop_duplicates().sort_values().diff().value_counts().head())   # resolución
print(fc["publishTime"].drop_duplicates().sort_values().diff().value_counts().head()) # frecuencia de publicación
print(fc.groupby("startTime").size().describe())       # pronósticos por slot
print("duplicados:", fc.duplicated(["startTime", "publishTime"]).sum())

count    25112.000000
mean        21.497335
std         22.045902
min        -25.500000
25%          3.500000
50%         21.500000
75%         39.500000
max         66.500000
Name: horizon_h, dtype: float64
horizonte negativo: 5161
startTime
0 days 01:00:00    1080
Name: count, dtype: int64
publishTime
0 days 02:00:00    128
0 days 04:00:00    127
0 days 03:00:00     86
0 days 02:55:00      1
0 days 03:05:00      1
Name: count, dtype: int64
count    1081.000000
mean       23.230342
std         3.999959
min         8.000000
25%        24.000000
50%        24.000000
75%        24.000000
max        32.000000
dtype: float64
duplicados: 0


In [18]:
# 1. Actuals a resolución horaria (promedio de los dos slots de 30 min)
act_h = (
    actual.set_index("startTime")["generation"]
    .resample("1h").mean()
    .rename("actual").reset_index()
)

# 2. Para cada hora objetivo, último pronóstico publicado antes del corte (D-1 a las 09:00)
def decision_forecasts(fc, decision_hour=9):
    d = fc.rename(columns={"generation": "forecast"}).copy()
    d["target_date"] = d["startTime"].dt.floor("D")
    d["cutoff"] = d["target_date"] - pd.Timedelta(days=1) + pd.Timedelta(hours=decision_hour)
    d = d[d["publishTime"] <= d["cutoff"]]                    # solo lo que existía al decidir
    d = d.sort_values("publishTime").groupby("startTime").tail(1)  # el más reciente por hora
    d["horizon_h"] = (d["startTime"] - d["publishTime"]).dt.total_seconds() / 3600
    return d

dec = decision_forecasts(fc)

# 3. Unir con actuals y limitar a los días objetivo de enero (decisiones del 1 al 31 de enero)
df = dec.merge(act_h, on="startTime", how="inner")
lo, hi = pd.Timestamp("2026-01-02", tz="UTC"), pd.Timestamp("2026-02-02", tz="UTC")
df = df[(df["startTime"] >= lo) & (df["startTime"] < hi)].copy()
df["error"] = df["forecast"] - df["actual"]   # positivo = el pronóstico sobreestimó

In [19]:
print(len(df), df["startTime"].nunique())      # esperado: 744 (31 días x 24 h)
print(df["horizon_h"].describe())              # esperado: mínimo >= 15, máximo ~ 43
print(df[["forecast", "actual"]].isna().sum()) # esperado: 0
print(df[["forecast", "actual"]].corr())       # esperado: correlación alta (~0.9+)
print(df["error"].describe())

744 744
count    744.000000
mean      28.000000
std        6.926843
min       16.500000
25%       22.250000
50%       28.000000
75%       33.750000
max       39.500000
Name: horizon_h, dtype: float64
forecast    0
actual      0
dtype: int64
          forecast    actual
forecast  1.000000  0.930877
actual    0.930877  1.000000
count     744.000000
mean     1747.887097
std      1795.282331
min     -3550.000000
25%       717.500000
50%      1592.000000
75%      2775.125000
max      6870.000000
Name: error, dtype: float64


In [20]:
# 1. Niveles medios: ¿el pronóstico es un X% más alto, o el sesgo es constante?
print(df[["forecast", "actual"]].mean())
print((df["error"].sum() / df["actual"].sum()))        # sesgo relativo

# 2. Test de desfase: ¿mejora el error si muevo el actual unas horas?
for shift_h in [-2, -1, 0, 1, 2]:
    a = df.set_index("startTime")["actual"].shift(shift_h, freq="1h")
    tmp = df.set_index("startTime")["forecast"].to_frame().join(a.rename("a")).dropna()
    mae = (tmp["forecast"] - tmp["a"]).abs().mean()
    print(f"shift {shift_h:+d}h  MAE={mae:8.1f}  corr={tmp['forecast'].corr(tmp['a']):.3f}")

# 3. ¿El sesgo es constante en el mes o cambia por día?
print(df.groupby(df["startTime"].dt.date)["error"].mean().describe())

forecast    13031.444892
actual      11283.557796
dtype: float64
0.1549056714576722
shift -2h  MAE=  2211.1  corr=0.911
shift -1h  MAE=  2121.1  corr=0.924
shift +0h  MAE=  2038.9  corr=0.931
shift +1h  MAE=  1972.8  corr=0.933
shift +2h  MAE=  1954.8  corr=0.929
count      31.000000
mean     1747.887097
std      1507.982577
min      -730.062500
25%       824.812500
50%      1554.833333
75%      2246.187500
max      5374.791667
Name: error, dtype: float64


In [21]:
import numpy as np

e = df["error"]
ae = e.abs()

summary = {
    "n": len(e),
    "MAE": ae.mean(),
    "RMSE": np.sqrt((e**2).mean()),
    "bias_MW": e.mean(),
    "bias_%": e.sum() / df["actual"].sum() * 100,
    "MAE_%_of_mean_actual": ae.mean() / df["actual"].mean() * 100,
    "abs_err_P50": ae.quantile(0.50),
    "abs_err_P90": ae.quantile(0.90),
    "abs_err_P99": ae.quantile(0.99),
    "err_P5": e.quantile(0.05),    # cola de infraestimación
    "err_P95": e.quantile(0.95),   # cola de sobreestimación (la que perjudica al comprador)
}
print(pd.Series(summary).round(1))

# Efecto de quitar el sesgo: ¿cuánto del error es sistemático?
e_c = e - e.mean()
print("MAE sin sesgo:", e_c.abs().mean().round(1), " RMSE sin sesgo:", np.sqrt((e_c**2).mean()).round(1))

# Por hora del día
df["hour"] = df["startTime"].dt.hour
print(df.groupby("hour")["error"].agg(bias="mean", mae=lambda x: x.abs().mean()).round(0))

# Por horizonte (en tramos)
df["h_bin"] = pd.cut(df["horizon_h"], [15, 22, 28, 34, 40])
print(df.groupby("h_bin", observed=True)["error"].agg(bias="mean", mae=lambda x: x.abs().mean(), n="count").round(0))

# Por nivel de generación real
df["lvl"] = pd.qcut(df["actual"], 4, labels=["bajo", "medio-bajo", "medio-alto", "alto"])
print(df.groupby("lvl", observed=True)["error"].agg(bias="mean", mae=lambda x: x.abs().mean()).round(0))

n                        744.0
MAE                     2038.9
RMSE                    2504.8
bias_MW                 1747.9
bias_%                    15.5
MAE_%_of_mean_actual      18.1
abs_err_P50             1671.0
abs_err_P90             4097.9
abs_err_P99             6261.3
err_P5                 -1151.4
err_P95                 5283.4
dtype: float64
MAE sin sesgo: 1375.9  RMSE sin sesgo: 1794.1
        bias     mae
hour                
0     1953.0  2028.0
1     2133.0  2169.0
2     2254.0  2299.0
3     2322.0  2413.0
4     2335.0  2405.0
5     2358.0  2430.0
6     2312.0  2380.0
7     2139.0  2227.0
8     1903.0  2032.0
9     1854.0  1996.0
10    1731.0  1931.0
11    1570.0  1963.0
12    1388.0  1929.0
13    1331.0  1943.0
14    1328.0  1914.0
15    1247.0  1818.0
16    1303.0  1847.0
17    1320.0  1918.0
18    1348.0  1955.0
19    1389.0  1953.0
20    1436.0  1856.0
21    1544.0  1786.0
22    1636.0  1793.0
23    1815.0  1950.0
            bias     mae    n
h_bin                 

In [22]:
allf = fc.rename(columns={"generation": "forecast"}).merge(act_h, on="startTime")
allf["horizon_h"] = (allf["startTime"] - allf["publishTime"]).dt.total_seconds() / 3600
allf = allf[(allf["horizon_h"] > 0) & (allf["startTime"] >= lo) & (allf["startTime"] < hi)]
allf["error"] = allf["forecast"] - allf["actual"]

allf["h_bin"] = pd.cut(allf["horizon_h"], [0, 6, 12, 24, 36, 48, 72])
print(allf.groupby("h_bin", observed=True)["error"]
      .agg(bias="mean", mae=lambda x: x.abs().mean(), n="count").round(0))

            bias     mae     n
h_bin                         
(0, 6]    1453.0  1686.0  1446
(6, 12]   1528.0  1763.0  1456
(12, 24]  1716.0  1905.0  2954
(24, 36]  1718.0  2019.0  2976
(36, 48]  1905.0  2250.0  2945
(48, 72]  1403.0  2317.0  2510
